<a href="https://colab.research.google.com/github/brunacorreiade/ruf-/blob/main/analise_ingressantes_inep_2015.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
from pathlib import Path
from urllib.request import Request, urlopen
import hashlib
import shutil
import zipfile

import pandas as pd
from IPython.display import display


# ============================================================
# 1. CONFIGURAÇÃO
# ============================================================

ANO = 2015

URL_PAGINA_INEP = (
    "https://www.gov.br/inep/pt-br/acesso-a-informacao/"
    "dados-abertos/microdados/censo-da-educacao-superior"
)

URL_ZIP = (
    "https://download.inep.gov.br/microdados/"
    "microdados_censo_da_educacao_superior_2015.zip"
)

NOME_ZIP = "microdados_censo_da_educacao_superior_2015.zip"

PASTA_TRABALHO = Path("/content")
PASTA_TRABALHO.mkdir(parents=True, exist_ok=True)

caminho_zip = PASTA_TRABALHO / NOME_ZIP


# ============================================================
# 2. BAIXAR O ZIP DO INEP
# ============================================================

def baixar_arquivo(url, destino):
    temporario = destino.with_suffix(destino.suffix + ".part")

    try:
        requisicao = Request(
            url,
            headers={"User-Agent": "Mozilla/5.0"}
        )

        with urlopen(requisicao, timeout=300) as resposta:
            tamanho = int(
                resposta.headers.get("Content-Length", 0)
            ) or None

            baixado = 0
            proximo_aviso = 50 * 1024 * 1024

            with open(temporario, "wb") as arquivo:
                while True:
                    bloco = resposta.read(1024 * 1024)

                    if not bloco:
                        break

                    arquivo.write(bloco)
                    baixado += len(bloco)

                    if baixado >= proximo_aviso:
                        if tamanho:
                            print(
                                f"Baixados {baixado / 1024**2:.0f} de "
                                f"{tamanho / 1024**2:.0f} MiB..."
                            )
                        else:
                            print(
                                f"Baixados {baixado / 1024**2:.0f} MiB..."
                            )

                        proximo_aviso += 50 * 1024 * 1024

        temporario.replace(destino)

    except Exception:
        temporario.unlink(missing_ok=True)
        raise


if caminho_zip.exists():
    print(f"ZIP já disponível: {caminho_zip.name}")

else:
    try:
        print("Baixando o ZIP oficial do Inep...")
        baixar_arquivo(URL_ZIP, caminho_zip)
        print(f"Download concluído: {caminho_zip.name}")

    except Exception as erro_download:
        print("O download automático não funcionou.")
        print(f"Motivo: {erro_download}")
        print("Selecione manualmente o ZIP oficial de 2015.")

        from google.colab import files

        arquivos_enviados = files.upload()

        arquivos_zip = [
            nome
            for nome in arquivos_enviados
            if nome.lower().endswith(".zip")
        ]

        if not arquivos_zip:
            raise FileNotFoundError(
                "Nenhum arquivo ZIP foi enviado."
            )

        nome_enviado = next(
            (
                nome
                for nome in arquivos_zip
                if "2015" in nome
            ),
            arquivos_zip[0]
        )

        caminho_zip.write_bytes(
            arquivos_enviados[nome_enviado]
        )

        print(f"ZIP salvo como: {caminho_zip.name}")


# ============================================================
# 3. REGISTRAR URL, TAMANHO E SHA-256
# ============================================================

def calcular_sha256(caminho):
    resumo = hashlib.sha256()

    with open(caminho, "rb") as arquivo:
        for bloco in iter(
            lambda: arquivo.read(1024 * 1024),
            b""
        ):
            resumo.update(bloco)

    return resumo.hexdigest()


sha256_zip = calcular_sha256(caminho_zip)
tamanho_zip = caminho_zip.stat().st_size

metadados_fonte = pd.DataFrame([{
    "ano": ANO,
    "fonte": "Inep — Censo da Educação Superior",
    "pagina_oficial": URL_PAGINA_INEP,
    "url_download": URL_ZIP,
    "nome_arquivo": caminho_zip.name,
    "tamanho_bytes": tamanho_zip,
    "sha256": sha256_zip
}])

print("\nIDENTIFICAÇÃO DO ARQUIVO")
display(metadados_fonte)


# ============================================================
# 4. DEFINIR AS VARIÁVEIS
# ============================================================

colunas_idade = [
    "QT_ING_0_17",
    "QT_ING_18_24",
    "QT_ING_25_29",
    "QT_ING_30_34",
    "QT_ING_35_39",
    "QT_ING_40_49",
    "QT_ING_50_59",
    "QT_ING_60_MAIS"
]

colunas_30_mais = [
    "QT_ING_30_34",
    "QT_ING_35_39",
    "QT_ING_40_49",
    "QT_ING_50_59",
    "QT_ING_60_MAIS"
]

colunas_contagem = [
    "QT_ING",
    "QT_ING_DIURNO",
    "QT_ING_NOTURNO"
] + colunas_idade

colunas_utilizadas = [
    "NU_ANO_CENSO",
    "TP_DIMENSAO",
    "TP_CATEGORIA_ADMINISTRATIVA",
    "TP_REDE",
    "CO_IES",
    "CO_CURSO",
    "NO_CURSO",
    "CO_CINE_ROTULO",
    "NO_CINE_ROTULO",
    "TP_MODALIDADE_ENSINO"
] + colunas_contagem


# ============================================================
# 5. LOCALIZAR E ABRIR O CSV
# ============================================================

with zipfile.ZipFile(caminho_zip, "r") as arquivo_zip:
    arquivo_cursos = next(
        (
            nome
            for nome in arquivo_zip.namelist()
            if nome.upper().endswith(
                "MICRODADOS_CADASTRO_CURSOS_2015.CSV"
            )
        ),
        None
    )

    if arquivo_cursos is None:
        raise FileNotFoundError(
            "O CSV de cursos não foi encontrado no ZIP."
        )

    with arquivo_zip.open(arquivo_cursos) as arquivo_csv:
        cabecalho = pd.read_csv(
            arquivo_csv,
            sep=";",
            encoding="latin1",
            nrows=0
        ).columns

    colunas_ausentes = sorted(
        set(colunas_utilizadas) - set(cabecalho)
    )

    if colunas_ausentes:
        raise KeyError(
            "Colunas esperadas não encontradas: "
            + ", ".join(colunas_ausentes)
        )

    with arquivo_zip.open(arquivo_cursos) as arquivo_csv:
        dados = pd.read_csv(
            arquivo_csv,
            sep=";",
            encoding="latin1",
            usecols=colunas_utilizadas,
            na_values=["."],
            low_memory=False,
            dtype={
                "CO_IES": "string",
                "CO_CURSO": "string",
                "CO_CINE_ROTULO": "string"
            }
        )

print(f"\nArquivo interno: {arquivo_cursos}")
print(f"Linhas: {len(dados):,}")
print(f"Colunas utilizadas: {len(dados.columns)}")


# ============================================================
# 6. LIMPEZA E CRIAÇÃO DAS VARIÁVEIS
# ============================================================

for coluna in colunas_contagem:
    dados[coluna] = (
        pd.to_numeric(
            dados[coluna],
            errors="coerce"
        )
        .fillna(0)
        .astype("int64")
    )

dados["CO_CINE_ROTULO"] = (
    dados["CO_CINE_ROTULO"]
    .str.replace('"', "", regex=False)
    .str.strip()
)

mapa_modalidade = {
    1: "Presencial",
    2: "EaD"
}

mapa_rede = {
    1: "Pública",
    2: "Privada"
}

mapa_categoria = {
    1: "Pública federal",
    2: "Pública estadual",
    3: "Pública municipal",
    4: "Privada com fins lucrativos",
    5: "Privada sem fins lucrativos",
    6: "Privada particular",
    7: "Especial",
    8: "Privada comunitária",
    9: "Privada confessional"
}

dados["modalidade"] = (
    dados["TP_MODALIDADE_ENSINO"]
    .map(mapa_modalidade)
)

dados["rede"] = (
    dados["TP_REDE"]
    .map(mapa_rede)
)

dados["categoria_administrativa"] = (
    dados["TP_CATEGORIA_ADMINISTRATIVA"]
    .map(mapa_categoria)
)

if dados["modalidade"].isna().any():
    codigos = sorted(
        dados.loc[
            dados["modalidade"].isna(),
            "TP_MODALIDADE_ENSINO"
        ].unique()
    )

    raise ValueError(
        f"Código de modalidade não mapeado: {codigos}"
    )

if dados["rede"].isna().any():
    codigos = sorted(
        dados.loc[
            dados["rede"].isna(),
            "TP_REDE"
        ].unique()
    )

    raise ValueError(
        f"Código de rede não mapeado: {codigos}"
    )

dados["ingressantes_30_mais"] = (
    dados[colunas_30_mais].sum(axis=1)
)


# ============================================================
# 7. CHECAGENS
# ============================================================

total_ingressantes = int(
    dados["QT_ING"].sum()
)

total_faixas = int(
    dados[colunas_idade].sum().sum()
)

total_30_mais = int(
    dados["ingressantes_30_mais"].sum()
)

linhas_inconsistentes = int(
    (
        dados[colunas_idade].sum(axis=1)
        != dados["QT_ING"]
    ).sum()
)

checagens = pd.DataFrame([
    {
        "checagem": "Total de ingressantes",
        "valor": total_ingressantes,
        "resultado_esperado": 2_922_400
    },
    {
        "checagem": "Ingressantes com 30 anos ou mais",
        "valor": total_30_mais,
        "resultado_esperado": 784_622
    },
    {
        "checagem": "Soma das faixas etárias",
        "valor": total_faixas,
        "resultado_esperado": total_ingressantes
    },
    {
        "checagem": "Linhas em que idade não soma QT_ING",
        "valor": linhas_inconsistentes,
        "resultado_esperado": 0
    }
])

checagens["status"] = checagens.apply(
    lambda linha: (
        "OK"
        if linha["valor"]
        == linha["resultado_esperado"]
        else "REVISAR"
    ),
    axis=1
)

print("\nCHECAGENS")
display(checagens)

assert (checagens["status"] == "OK").all(), (
    "Uma ou mais checagens falharam. "
    "Não use os resultados antes de revisar."
)


# ============================================================
# 8. FUNÇÃO PARA EXIBIR AS TABELAS
# ============================================================

def exibir_tabela(
    tabela,
    colunas_percentuais=()
):
    copia = tabela.copy()

    for coluna in colunas_percentuais:
        if coluna in copia.columns:
            copia[coluna] = copia[coluna].map(
                lambda valor: (
                    f"{valor:.2%}"
                    if pd.notna(valor)
                    else ""
                )
            )

    display(copia)


# ============================================================
# 9. RESULTADOS GERAIS E FAIXAS ETÁRIAS
# ============================================================

nomes_faixas = {
    "QT_ING_0_17": "Até 17",
    "QT_ING_18_24": "18 a 24",
    "QT_ING_25_29": "25 a 29",
    "QT_ING_30_34": "30 a 34",
    "QT_ING_35_39": "35 a 39",
    "QT_ING_40_49": "40 a 49",
    "QT_ING_50_59": "50 a 59",
    "QT_ING_60_MAIS": "60 ou mais"
}

ordem_faixas = list(
    nomes_faixas.values()
)

idade_brasil = (
    dados[colunas_idade]
    .sum()
    .rename_axis("variavel")
    .reset_index(name="ingressantes")
)

idade_brasil["faixa_etaria"] = (
    idade_brasil["variavel"]
    .map(nomes_faixas)
)

idade_brasil["percentual_total"] = (
    idade_brasil["ingressantes"]
    / total_ingressantes
)

idade_brasil = idade_brasil[
    [
        "faixa_etaria",
        "ingressantes",
        "percentual_total"
    ]
]

resumo_geral = pd.DataFrame([{
    "ano": ANO,
    "total_ingressantes": total_ingressantes,
    "ingressantes_30_mais": total_30_mais,
    "percentual_30_mais": (
        total_30_mais / total_ingressantes
    )
}])

print("\nRESUMO GERAL")
exibir_tabela(
    resumo_geral,
    ["percentual_30_mais"]
)

print("\nINGRESSANTES POR FAIXA ETÁRIA")
exibir_tabela(
    idade_brasil,
    ["percentual_total"]
)


# ============================================================
# 10. MODALIDADE E REDE
# ============================================================

resumo_30_modalidade_rede = (
    dados
    .groupby(
        ["modalidade", "rede"],
        as_index=False
    )
    .agg(
        total_ingressantes=("QT_ING", "sum"),
        ingressantes_30_mais=(
            "ingressantes_30_mais",
            "sum"
        )
    )
)

resumo_30_modalidade_rede[
    "percentual_30_mais"
] = (
    resumo_30_modalidade_rede[
        "ingressantes_30_mais"
    ]
    / resumo_30_modalidade_rede[
        "total_ingressantes"
    ]
)

resumo_30_modalidade_rede[
    "participacao_total_30_mais"
] = (
    resumo_30_modalidade_rede[
        "ingressantes_30_mais"
    ]
    / total_30_mais
)

resumo_modalidade = (
    dados
    .groupby(
        "modalidade",
        as_index=False
    )
    .agg(
        total_ingressantes=("QT_ING", "sum"),
        ingressantes_30_mais=(
            "ingressantes_30_mais",
            "sum"
        )
    )
)

resumo_modalidade[
    "percentual_30_mais"
] = (
    resumo_modalidade[
        "ingressantes_30_mais"
    ]
    / resumo_modalidade[
        "total_ingressantes"
    ]
)

resumo_modalidade[
    "participacao_total_ingressantes"
] = (
    resumo_modalidade[
        "total_ingressantes"
    ]
    / total_ingressantes
)

resumo_modalidade[
    "participacao_total_30_mais"
] = (
    resumo_modalidade[
        "ingressantes_30_mais"
    ]
    / total_30_mais
)

resumo_rede = (
    dados
    .groupby(
        "rede",
        as_index=False
    )
    .agg(
        total_ingressantes=("QT_ING", "sum"),
        ingressantes_30_mais=(
            "ingressantes_30_mais",
            "sum"
        )
    )
)

resumo_rede[
    "percentual_30_mais"
] = (
    resumo_rede[
        "ingressantes_30_mais"
    ]
    / resumo_rede[
        "total_ingressantes"
    ]
)

resumo_rede[
    "participacao_total_30_mais"
] = (
    resumo_rede[
        "ingressantes_30_mais"
    ]
    / total_30_mais
)

colunas_percentuais = [
    "percentual_30_mais",
    "participacao_total_ingressantes",
    "participacao_total_30_mais"
]

print("\nRESUMO POR MODALIDADE")
exibir_tabela(
    resumo_modalidade,
    colunas_percentuais
)

print("\nRESUMO POR REDE")
exibir_tabela(
    resumo_rede,
    colunas_percentuais
)

print("\nMODALIDADE × REDE")
exibir_tabela(
    resumo_30_modalidade_rede,
    colunas_percentuais
)


# ============================================================
# 11. CATEGORIA ADMINISTRATIVA
# ============================================================

resumo_categoria = (
    dados
    .groupby(
        "categoria_administrativa",
        as_index=False
    )
    .agg(
        total_ingressantes=("QT_ING", "sum"),
        ingressantes_30_mais=(
            "ingressantes_30_mais",
            "sum"
        )
    )
)

resumo_categoria[
    "percentual_30_mais"
] = (
    resumo_categoria[
        "ingressantes_30_mais"
    ]
    / resumo_categoria[
        "total_ingressantes"
    ]
)

resumo_categoria[
    "participacao_total_30_mais"
] = (
    resumo_categoria[
        "ingressantes_30_mais"
    ]
    / total_30_mais
)

print("\nCATEGORIA ADMINISTRATIVA")
exibir_tabela(
    resumo_categoria,
    colunas_percentuais
)


# ============================================================
# 12. FAIXA ETÁRIA × MODALIDADE × REDE
# ============================================================

faixa_modalidade_rede = dados.melt(
    id_vars=[
        "modalidade",
        "rede"
    ],
    value_vars=colunas_idade,
    var_name="variavel_idade",
    value_name="ingressantes"
)

faixa_modalidade_rede[
    "faixa_etaria"
] = (
    faixa_modalidade_rede[
        "variavel_idade"
    ].map(nomes_faixas)
)

faixa_modalidade_rede = (
    faixa_modalidade_rede
    .groupby(
        [
            "faixa_etaria",
            "modalidade",
            "rede"
        ],
        as_index=False
    )["ingressantes"]
    .sum()
)

faixa_modalidade_rede[
    "percentual_no_grupo"
] = (
    faixa_modalidade_rede
    .groupby(
        ["modalidade", "rede"]
    )["ingressantes"]
    .transform(
        lambda valores: (
            valores / valores.sum()
        )
    )
)

faixa_modalidade_rede[
    "faixa_etaria"
] = pd.Categorical(
    faixa_modalidade_rede[
        "faixa_etaria"
    ],
    categories=ordem_faixas,
    ordered=True
)

faixa_modalidade_rede = (
    faixa_modalidade_rede
    .sort_values(
        [
            "faixa_etaria",
            "modalidade",
            "rede"
        ]
    )
    .reset_index(drop=True)
)

print("\nFAIXA ETÁRIA × MODALIDADE × REDE")
exibir_tabela(
    faixa_modalidade_rede,
    ["percentual_no_grupo"]
)


# ============================================================
# 13. PEDAGOGIA E SERVIÇO SOCIAL
# ============================================================

cursos_pauta = {
    "0113P01": "Pedagogia",
    "0923S01": "Serviço Social"
}

dados["curso_pauta"] = (
    dados["CO_CINE_ROTULO"]
    .map(cursos_pauta)
)

dados_cursos = dados[
    dados["curso_pauta"].notna()
].copy()

resumo_cursos_geral = (
    dados_cursos
    .groupby(
        "curso_pauta",
        as_index=False
    )
    .agg(
        total_ingressantes=("QT_ING", "sum"),
        ingressantes_30_mais=(
            "ingressantes_30_mais",
            "sum"
        )
    )
)

resumo_cursos_geral[
    "percentual_30_mais"
] = (
    resumo_cursos_geral[
        "ingressantes_30_mais"
    ]
    / resumo_cursos_geral[
        "total_ingressantes"
    ]
)

resumo_cursos = (
    dados_cursos
    .groupby(
        [
            "curso_pauta",
            "modalidade",
            "rede"
        ],
        as_index=False
    )
    .agg(
        total_ingressantes=("QT_ING", "sum"),
        ingressantes_30_mais=(
            "ingressantes_30_mais",
            "sum"
        )
    )
)

resumo_cursos[
    "percentual_30_mais"
] = (
    resumo_cursos[
        "ingressantes_30_mais"
    ]
    .div(
        resumo_cursos[
            "total_ingressantes"
        ].replace(0, pd.NA)
    )
)

faixas_cursos = dados_cursos.melt(
    id_vars=[
        "curso_pauta",
        "modalidade",
        "rede"
    ],
    value_vars=colunas_idade,
    var_name="variavel_idade",
    value_name="ingressantes"
)

faixas_cursos[
    "faixa_etaria"
] = (
    faixas_cursos[
        "variavel_idade"
    ].map(nomes_faixas)
)

faixas_cursos = (
    faixas_cursos
    .groupby(
        [
            "curso_pauta",
            "faixa_etaria",
            "modalidade",
            "rede"
        ],
        as_index=False
    )["ingressantes"]
    .sum()
)

faixas_cursos[
    "faixa_etaria"
] = pd.Categorical(
    faixas_cursos[
        "faixa_etaria"
    ],
    categories=ordem_faixas,
    ordered=True
)

faixas_cursos = (
    faixas_cursos
    .sort_values(
        [
            "curso_pauta",
            "faixa_etaria",
            "modalidade",
            "rede"
        ]
    )
    .reset_index(drop=True)
)

print("\nRESUMO GERAL DOS CURSOS")
exibir_tabela(
    resumo_cursos_geral,
    ["percentual_30_mais"]
)

print("\nCURSOS POR MODALIDADE E REDE")
exibir_tabela(
    resumo_cursos,
    ["percentual_30_mais"]
)

print("\nCURSOS POR FAIXA ETÁRIA")
display(faixas_cursos)


# ============================================================
# 14. TURNOS DOS CURSOS PRESENCIAIS
# ============================================================
# A base não permite cruzar diretamente faixa etária e turno.

presencial = dados[
    dados["modalidade"] == "Presencial"
]

turnos_presenciais = pd.DataFrame({
    "turno": [
        "Diurno",
        "Noturno"
    ],
    "ingressantes": [
        int(
            presencial[
                "QT_ING_DIURNO"
            ].sum()
        ),
        int(
            presencial[
                "QT_ING_NOTURNO"
            ].sum()
        )
    ]
})

turnos_presenciais[
    "percentual"
] = (
    turnos_presenciais[
        "ingressantes"
    ]
    / turnos_presenciais[
        "ingressantes"
    ].sum()
)

print("\nTURNOS — CURSOS PRESENCIAIS")
exibir_tabela(
    turnos_presenciais,
    ["percentual"]
)


# ============================================================
# 15. CRIAR PASTAS DO PROJETO
# ============================================================

PASTA_PROJETO = (
    PASTA_TRABALHO
    / "universidade-mais-velha"
)

PASTA_RESULTADOS = (
    PASTA_PROJETO
    / "resultados"
)

PASTA_RESULTADOS.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 16. EXPORTAR OS CSVs
# ============================================================

tabelas = {
    "checagens_2015.csv":
        checagens,

    "metadados_fonte_2015.csv":
        metadados_fonte,

    "resumo_geral_2015.csv":
        resumo_geral,

    "idade_brasil_2015.csv":
        idade_brasil,

    "resumo_modalidade_2015.csv":
        resumo_modalidade,

    "resumo_rede_2015.csv":
        resumo_rede,

    "30_mais_modalidade_rede_2015.csv":
        resumo_30_modalidade_rede,

    "categoria_administrativa_2015.csv":
        resumo_categoria,

    "faixa_modalidade_rede_2015.csv":
        faixa_modalidade_rede,

    "resumo_cursos_geral_2015.csv":
        resumo_cursos_geral,

    "pedagogia_servico_social_2015.csv":
        resumo_cursos,

    "cursos_por_faixa_2015.csv":
        faixas_cursos,

    "turnos_presenciais_2015.csv":
        turnos_presenciais
}

for nome_arquivo, tabela in tabelas.items():
    tabela.to_csv(
        PASTA_RESULTADOS / nome_arquivo,
        index=False,
        encoding="utf-8-sig"
    )


# ============================================================
# 17. EXPORTAR EXCEL FORMATADO
# ============================================================

caminho_excel = (
    PASTA_RESULTADOS
    / "resultados_inep_2015.xlsx"
)

abas_excel = {
    "Checagens":
        checagens,

    "Fonte":
        metadados_fonte,

    "Resumo geral":
        resumo_geral,

    "Faixas etárias":
        idade_brasil,

    "Modalidade":
        resumo_modalidade,

    "Rede":
        resumo_rede,

    "30+ modalidade rede":
        resumo_30_modalidade_rede,

    "Categoria administrativa":
        resumo_categoria,

    "Faixa modalidade rede":
        faixa_modalidade_rede,

    "Resumo dos cursos":
        resumo_cursos_geral,

    "Cursos 30+":
        resumo_cursos,

    "Cursos por faixa":
        faixas_cursos,

    "Turnos presenciais":
        turnos_presenciais
}

with pd.ExcelWriter(
    caminho_excel,
    engine="openpyxl"
) as writer:

    for nome_aba, tabela in abas_excel.items():
        tabela.to_excel(
            writer,
            sheet_name=nome_aba,
            index=False
        )

    from openpyxl.styles import (
        Alignment,
        Font,
        PatternFill
    )

    from openpyxl.utils import (
        get_column_letter
    )

    preenchimento = PatternFill(
        "solid",
        fgColor="1F4E78"
    )

    fonte = Font(
        color="FFFFFF",
        bold=True
    )

    for planilha in writer.book.worksheets:
        planilha.freeze_panes = "A2"
        planilha.auto_filter.ref = (
            planilha.dimensions
        )

        for celula in planilha[1]:
            celula.fill = preenchimento
            celula.font = fonte
            celula.alignment = Alignment(
                horizontal="center"
            )

        for indice, coluna in enumerate(
            planilha.iter_cols(),
            start=1
        ):
            maior = max(
                len(str(celula.value))
                if celula.value is not None
                else 0
                for celula in coluna
            )

            planilha.column_dimensions[
                get_column_letter(indice)
            ].width = min(
                max(maior + 2, 12),
                45
            )

        cabecalhos = {
            celula.value: celula.column
            for celula in planilha[1]
        }

        for nome_coluna, indice in (
            cabecalhos.items()
        ):
            if nome_coluna and (
                "percentual"
                in str(nome_coluna)
                or "participacao"
                in str(nome_coluna)
            ):
                for linha in range(
                    2,
                    planilha.max_row + 1
                ):
                    planilha.cell(
                        linha,
                        indice
                    ).number_format = "0.00%"


# ============================================================
# 18. CRIAR README
# ============================================================

def inteiro_br(valor):
    return (
        f"{int(valor):,}"
        .replace(",", ".")
    )


def percentual_br(valor):
    return (
        f"{valor:.2%}"
        .replace(".", ",")
    )


texto_readme = f"""# Universidade fica mais velha

Análise dos ingressantes do ensino superior brasileiro com base nos microdados do Censo da Educação Superior de 2015, do Inep.

## Resultados gerais de 2015

- Total de ingressantes: {inteiro_br(total_ingressantes)}
- Ingressantes com 30 anos ou mais: {inteiro_br(total_30_mais)}
- Participação dos ingressantes 30+: {percentual_br(total_30_mais / total_ingressantes)}

## Fonte

Inep — Censo da Educação Superior de 2015.

Os microdados podem ser baixados na página oficial:

{URL_PAGINA_INEP}

O notebook registra a URL, o tamanho e o SHA-256 do ZIP utilizado. O arquivo bruto não foi incluído no repositório.

## Metodologia

A análise utiliza o arquivo `MICRODADOS_CADASTRO_CURSOS_2015.CSV`, agregado por curso.

O total de ingressantes é obtido por meio da variável `QT_ING`.

Os ingressantes com 30 anos ou mais correspondem à soma de:

- `QT_ING_30_34`
- `QT_ING_35_39`
- `QT_ING_40_49`
- `QT_ING_50_59`
- `QT_ING_60_MAIS`

Pedagogia e Serviço Social foram identificados pelos códigos Cine `0113P01` e `0923S01`, respectivamente.

## Limitações

A base pública de 2015 está agregada por curso. Os resultados se referem a ingressos ou vínculos de ingressantes, e não necessariamente a pessoas únicas.

Não é possível cruzar diretamente faixa etária e turno, pois essas informações aparecem como totais marginais separados.

Os resultados de 2015, isoladamente, não permitem concluir que a expansão da EaD provocou o aumento da participação dos ingressantes mais velhos. Essa hipótese deve ser testada por meio da comparação com 2024.
"""

(PASTA_PROJETO / "README.md").write_text(
    texto_readme,
    encoding="utf-8"
)


# ============================================================
# 19. COMPACTAR E BAIXAR OS RESULTADOS
# ============================================================

caminho_pacote = shutil.make_archive(
    str(
        PASTA_TRABALHO
        / "resultados_inep_2015"
    ),
    "zip",
    root_dir=PASTA_PROJETO
)

print("\nANÁLISE CONCLUÍDA")
print(f"CSVs: {PASTA_RESULTADOS}")
print(f"Excel: {caminho_excel}")
print(f"Pacote: {caminho_pacote}")

from google.colab import files

files.download(caminho_pacote)

ZIP já disponível: microdados_censo_da_educacao_superior_2015.zip

IDENTIFICAÇÃO DO ARQUIVO


,ano,fonte,pagina_oficial,url_download,nome_arquivo,tamanho_bytes,sha256
0,2015,Inep — Censo da Educação Superior,https://www.gov.br/inep/pt-br/acesso-a-informa...,https://download.inep.gov.br/microdados/microd...,microdados_censo_da_educacao_superior_2015.zip,8400033,e1559a328c494acaebfe03a03e8f9f7666735354649ddc...



Arquivo interno: Microdados do Censo da Educaç╞o Superior 2015/dados/MICRODADOS_CADASTRO_CURSOS_2015.CSV
Linhas: 81,156
Colunas utilizadas: 21

CHECAGENS


,checagem,valor,resultado_esperado,status
0,Total de ingressantes,2922400,2922400,OK
1,Ingressantes com 30 anos ou mais,784622,784622,OK
2,Soma das faixas etárias,2922400,2922400,OK
3,Linhas em que idade não soma QT_ING,0,0,OK



RESUMO GERAL


...


TURNOS — CURSOS PRESENCIAIS


,turno,ingressantes,percentual
0,Diurno,816484,36.65%
1,Noturno,1411357,63.35%



ANÁLISE CONCLUÍDA
CSVs: /content/universidade-mais-velha/resultados
Excel: /content/universidade-mais-velha/resultados/resultados_inep_2015.xlsx
Pacote: /content/resultados_inep_2015.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>